In [1]:
# Importing necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
# Loading the dataset
dataset = pd.read_csv("insurance_pre.csv")
dataset.head()

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520


In [3]:
# Categorical data - One-Hot encodinng
dataset = pd.get_dummies(dataset,drop_first=True,dtype=int)
dataset.head()

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0


In [4]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [5]:
# Feature selection
independent = dataset[['age', 'bmi', 'children', 'sex_male', 'smoker_yes']]
independent.head() 

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0


In [6]:
# Target or output
dependent = dataset[['charges']]
dependent.head()

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520


In [7]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independent, dependent, test_size = 0.25, random_state = 0)

In [8]:
#Data standardization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
# GridSearchCV model training
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
param_grid={'criterion':['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],'max_features': ['sqrt','log2'],'splitter':['best','random']}
grid=GridSearchCV(DecisionTreeRegressor(),param_grid,refit=True,verbose=3,n_jobs=-1)
grid.fit(X_train,y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(estimator=DecisionTreeRegressor(), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'friedman_mse',
                                       'absolute_error', 'poisson'],
                         'max_features': ['sqrt', 'log2'],
                         'splitter': ['best', 'random']},
             verbose=3)

In [10]:
# Best model parameters
print("The best parameters are: ",grid.best_params_)
print("The best score is: ",grid.best_score_)

The best parameters are:  {'criterion': 'squared_error', 'max_features': 'sqrt', 'splitter': 'best'}
The best score is:  0.6359406716910897


In [11]:
# Gridsearchcv results
result = grid.cv_results_
table = pd.DataFrame.from_dict(result)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.007998,0.002072,0.004598,0.001526,squared_error,sqrt,best,"{'criterion': 'squared_error', 'max_features':...",0.648148,0.544608,0.641253,0.666342,0.679352,0.635941,0.047598,1
1,0.005945,0.001849,0.003488,0.000881,squared_error,sqrt,random,"{'criterion': 'squared_error', 'max_features':...",0.653314,0.606297,0.550461,0.581932,0.598763,0.598153,0.033607,9
2,0.007579,0.001914,0.003162,0.000452,squared_error,log2,best,"{'criterion': 'squared_error', 'max_features':...",0.731127,0.563472,0.573468,0.611386,0.642004,0.624291,0.060293,3
3,0.005920,0.002469,0.003423,0.001329,squared_error,log2,random,"{'criterion': 'squared_error', 'max_features':...",0.588928,0.424384,0.688629,0.739561,0.685583,0.625417,0.111705,2
4,0.006230,0.000799,0.005565,0.003078,friedman_mse,sqrt,best,"{'criterion': 'friedman_mse', 'max_features': ...",0.753065,0.557937,0.649636,0.462625,0.694191,0.623491,0.102612,4
5,0.005586,0.001702,0.003678,0.001193,friedman_mse,sqrt,random,"{'criterion': 'friedman_mse', 'max_features': ...",0.673864,0.448607,0.688970,0.659276,0.618725,0.617888,0.087813,7
6,0.005779,0.001299,0.003409,0.001655,friedman_mse,log2,best,"{'criterion': 'friedman_mse', 'max_features': ...",0.611673,0.546731,0.320546,0.473375,0.667962,0.524057,0.120735,16
7,0.005973,0.002120,0.003724,0.001156,friedman_mse,log2,random,"{'criterion': 'friedman_mse', 'max_features': ...",0.687213,0.557736,0.650352,0.655873,0.542691,0.618773,0.057570,6
8,0.017489,0.002789,0.003668,0.001331,absolute_error,sqrt,best,"{'criterion': 'absolute_error', 'max_features'...",0.549436,0.353372,0.587470,0.541108,0.684440,0.543165,0.107691,15
9,0.013438,0.001711,0.002816,0.000836,absolute_error,sqrt,random,"{'criterion': 'absolute_error', 'max_features'...",0.632508,0.530661,0.520693,0.673028,0.467196,0.564817,0.076072,12


In [12]:
# user inputs
age_input=float(input("Age:"))
bmi_input=float(input("BMI:"))
children_input=float(input("Children:"))
sex_male_input=float(input("Sex Male 0 or 1:"))
smoker_yes_input=float(input("Smoker yes 0 or 1:"))

In [13]:
# Model Prediction
Future_Prediction=grid.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])
print('Future_Prediction = ',Future_Prediction)

Future_Prediction =  [15555.18875]
